# @title 1. Install clipsmith + GPU transcription dependencies
# Takes ~2 min on first run
!pip install -q "clipsmith[ollama]"
# GPU-capable faster-whisper (CUDA 11.x bundled with T4 Colab runtime)
!pip install -q faster-whisper nvidia-cublas-cu11 nvidia-cudnn-cu11
!ffmpeg -version | head -1
# Confirm GPU is available
import subprocess
result = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
print(f"GPU: {result.stdout.strip() or 'NOT FOUND — switch to T4 runtime'}")

In [ ]:
# @title 1. Install clipsmith + GPU transcription dependencies
# Takes ~2 min on first run
!pip install -q "git+https://github.com/ricardogr07/clipsmith.git[ollama]"
# GPU-capable faster-whisper (CUDA 11.x bundled with T4 Colab runtime)
!pip install -q faster-whisper nvidia-cublas-cu11 nvidia-cudnn-cu11
!ffmpeg -version | head -1
# Confirm GPU is available
import subprocess

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True
)
print(f"GPU: {result.stdout.strip() or 'NOT FOUND — switch to T4 runtime'}")

In [ ]:
# @title 2. Load Twitch credentials from Colab Secrets (optional)
import os
from google.colab import userdata

os.environ["TWITCH_CLIENT_ID"] = userdata.get("TWITCH_CLIENT_ID") or ""
os.environ["TWITCH_CLIENT_SECRET"] = userdata.get("TWITCH_CLIENT_SECRET") or ""

if not os.environ["TWITCH_CLIENT_ID"]:
    print("Note: Twitch credentials not set — existing-clip boost signals will be skipped")
else:
    print("Twitch credentials loaded.")

In [ ]:
# @title 3. Install Ollama server and pull model
# Pulling ~4.7 GB takes ~3–5 min on first run per session.
# Uncomment the Drive lines below to persist and skip re-download on future sessions.
import subprocess
import sys
import time
import os

OLLAMA_MODEL = "llama3.1:8b"  # alternatives: mistral:7b (~4.1 GB), phi3:mini (~2.3 GB)

# --- Optional: persist model on Drive ---
# from google.colab import drive
# drive.mount("/content/drive")
# os.makedirs("/content/drive/MyDrive/ollama_models", exist_ok=True)
# os.environ["OLLAMA_MODELS"] = "/content/drive/MyDrive/ollama_models"
# ----------------------------------------

print("Installing Ollama...")
subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print("Starting Ollama server...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

print(f"Pulling {OLLAMA_MODEL}...")
subprocess.run(["ollama", "pull", OLLAMA_MODEL], check=True)
subprocess.run(
    [sys.executable, "-m", "clipsmith", "check-ollama", "--config", "config.yaml"], check=False
)
print("Ollama ready.")

In [ ]:
# @title 4. Configure — set your VOD ID here
from pathlib import Path

VOD_ID = "2758856582"  # <-- paste your Twitch VOD ID
LANGUAGE = "es"  # ISO 639-1 — change to 'en' for English streams
WHISPER_MODEL = "medium"  # medium is fast + accurate on T4; large-v3 for best quality
MAX_CANDIDATES = 20

config_yaml = f"""\
work_dir: /content/work
out_dir:  /content/out

llm:
  provider: ollama
  model_ollama: {OLLAMA_MODEL}

transcribe:
  model: {WHISPER_MODEL}
  compute_type: float16
  language: {LANGUAGE}
  chunk_minutes: 10
  max_workers: 2

clip:
  min_seconds: 15
  max_seconds: 30
  preroll_s: 25
  postroll_s: 10

reframe:
  mode: none

caption:
  enabled: false
"""

Path("/content/work").mkdir(parents=True, exist_ok=True)
Path("/content/out").mkdir(parents=True, exist_ok=True)
Path("config.yaml").write_text(config_yaml)
print(f"Config written — VOD: {VOD_ID}, Whisper: {WHISPER_MODEL}, LLM: ollama/{OLLAMA_MODEL}")

In [ ]:
# @title 5. Run the full pipeline (download → transcribe → select → cut)
# Re-running skips stages that already have cached output files.
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "clipsmith",
        "run-vod",
        VOD_ID,
        "--config",
        "config.yaml",
        "--max-candidates",
        str(MAX_CANDIDATES),
        "-v",
    ],
    check=True,
)

In [ ]:
# @title 6. Preview clips inline
from IPython.display import Video, display
from pathlib import Path

clips = sorted((Path("/content/out") / VOD_ID).glob("clip_*.mp4"))

if not clips:
    print("No clips found. Did step 5 complete successfully?")
else:
    print(f"Found {len(clips)} clip(s):")
    for mp4 in clips:
        print(f"  {mp4.name}")
        display(Video(str(mp4), embed=True, width=360))

In [ ]:
# @title 7. Download clips to your computer
from google.colab import files
from pathlib import Path

for mp4 in sorted((Path("/content/out") / VOD_ID).glob("clip_*.mp4")):
    files.download(str(mp4))